In [1]:
import os
import matplotlib.pyplot as plt
import torch
from nnfabrik.builder import get_data, get_trainer

from model import stacked_core_full_gauss_readout
from trainer import standard_trainer

from autoencoder import Autoenc

random_seed = 42

device = "cuda:7"
torch.cuda.set_device(device)

In [2]:
basepath = "/srv/user/polina/sensorium/sensorium/notebooks/data/"

# as filenames, we'll select all 7 datasets
filenames = [
    os.path.join(basepath, file) for file in os.listdir(basepath) if ".zip" in file
]


dataset_fn = "sensorium.datasets.static_loaders"
dataset_config = {
    "paths": filenames,
    "normalize": True,
    "include_behavior": False,
    "include_eye_position": True,
    "batch_size": 128,
    "scale": 0.25,
}

dataloaders = get_data(dataset_fn, dataset_config)
data_keys = list(dataloaders['train'].keys())

In [3]:
model_config = {
    "pad_input": False,
    "stack": -1,
    "layers": 4,
    "input_kern": 9,
    "gamma_input": 6.3831,
    "gamma_readout": 0.0076,
    "hidden_kern": 7,
    "hidden_channels": 64,
    "depth_separable": True,
    "grid_mean_predictor": {
        "type": "cortex",
        "input_dimensions": 2,
        "hidden_layers": 1,
        "hidden_features": 30,
        "final_tanh": True,
    },
    "init_sigma": 0.1,
    "init_mu_range": 0.3,
    "gauss_type": "full",
    "shifter": True, 
    "autoencoder": None, 
}

model = stacked_core_full_gauss_readout(dataloaders, random_seed, **model_config)

/srv/user/nathanpaul.soeding/private_neuropredictors/neuralpredictors/layers/readouts/base.py:74: UserWarning: Use of 'gamma_readout' is deprecated. Use 'feature_reg_weight' instead. If 'feature_reg_weight' is defined, 'gamma_readout' is ignored
  warnings.warn(
/srv/user/nathanpaul.soeding/private_neuropredictors/neuralpredictors/layers/readouts/base.py:95: UserWarning: Readout is NOT initialized with mean activity but with 0!
  warnings.warn("Readout is NOT initialized with mean activity but with 0!")


In [4]:
trainer_fn = "sensorium.training.standard_trainer"
 
trainer_config = {'max_iter': 200,
                 'verbose': False,
                 'lr_decay_steps': 4,
                 'avg_loss': False,
                 'lr_init': 0.009,
                 'device': device, 
                 }

trainer = get_trainer(trainer_fn=trainer_fn, 
                     trainer_config=trainer_config)

In [5]:
#validation_score, trainer_output, state_dict = trainer(model, dataloaders, seed=42)
validation_score, trainer_output, state_dict = standard_trainer(
    model, 
    dataloaders, 
    seed=42, 
    **trainer_config
)

torch.save(model.state_dict(), 'model_weights.pth')

/user/nathanpaul.soeding/miniconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
Epoch 88: 100%|██████████| 252/252 [00:22<00:00, 11.33it/s]


In [7]:
validation_score

np.float32(0.32743725)

In [5]:
model.load_state_dict(torch.load('checkpoints/base/model_weights0.pth'))

<All keys matched successfully>

In [6]:
# select all parameters that are not autoencoder
freeze_params = ['core', 'readout', 'shifter']

# copy parameters and freeze them
for name, param in model.named_parameters():
    param.requires_grad = not any(word in name for word in freeze_params)

In [7]:
autoencoder = Autoenc(
    input_dim=64, 
    latent_dim=16, 
    hidden_layers=1, 
    hidden_dims=128, 
    batch_norm=True, 
    nonlinearity='GELU'
)

for data_key in data_keys:
    model.readout[data_key].autoencoder = autoencoder

In [8]:
[(name, param.shape, param.requires_grad) for name, param in model.named_parameters()]

[('core.features.layer0.conv.weight', torch.Size([64, 4, 9, 9]), False),
 ('core.features.layer0.norm.weight', torch.Size([64]), False),
 ('core.features.layer0.norm.bias', torch.Size([64]), False),
 ('core.features.layer1.ds_conv.in_depth_conv.weight',
  torch.Size([64, 64, 1, 1]),
  False),
 ('core.features.layer1.ds_conv.in_depth_conv.bias', torch.Size([64]), False),
 ('core.features.layer1.ds_conv.spatial_conv.weight',
  torch.Size([64, 1, 7, 7]),
  False),
 ('core.features.layer1.ds_conv.spatial_conv.bias', torch.Size([64]), False),
 ('core.features.layer1.ds_conv.out_depth_conv.weight',
  torch.Size([64, 64, 1, 1]),
  False),
 ('core.features.layer1.ds_conv.out_depth_conv.bias', torch.Size([64]), False),
 ('core.features.layer1.norm.weight', torch.Size([64]), False),
 ('core.features.layer1.norm.bias', torch.Size([64]), False),
 ('core.features.layer2.ds_conv.in_depth_conv.weight',
  torch.Size([64, 64, 1, 1]),
  False),
 ('core.features.layer2.ds_conv.in_depth_conv.bias', torch.

In [ ]:
core_lr = 1e-10
autoencoder_lr = 1e-10
readout_lr = 1e-5
shifter_lr = 1e-5

''' 
optimizer = torch.optim.Adam([
    #{'params': model_autoenc.core.parameters(), 'lr': core_lr}, 
    {'params': model.readout[data_key].autoencoder.parameters(), 'lr': autoencoder_lr},
    #{'params': model_autoenc.readout[data_key].sigma, 'lr': readout_lr},
    #{'params': model_autoenc.readout[data_key]._features, 'lr': readout_lr},
    #{'params': model_autoenc.readout[data_key].bias, 'lr': readout_lr},
    #{'params': model_autoenc.readout[data_key].mu_transform.parameters(), 'lr': readout_lr},
    #{'params': model_autoenc.shifter.parameters(), 'lr': shifter_lr},
])
'''
trainer_config = {
    'max_iter': 5,
    'verbose': False,
    'lr_decay_steps': 4,
    'avg_loss': False,
    'lr_init': 0.009,
    'device': device, 
    #'optimizer': optimizer, 
}

validation_score, trainer_output, state_dict = standard_trainer(
    model, 
    dataloaders, 
    seed=42, 
    **trainer_config
)

torch.save(autoencoder.state_dict(), 'autoencoder_weights.pth')

/user/nathanpaul.soeding/miniconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
Epoch 1:  49%|████▉     | 124/252 [00:35<00:36,  3.51it/s]

In [10]:
validation_score

np.float32(0.3247073)

In [12]:
torch.save(autoencoder.state_dict(), 'autoencoder_weights.pth')

In [13]:
autoencoder.load_state_dict(torch.load('autoencoder_weights.pth'))

<All keys matched successfully>

In [ ]:
from sensorium.utility.scores import get_correlations

for data_key in data_keys:
    model.readout[data_key].autoencoder = None

model.eval()

# Compute avg validation and test correlation
validation_correlation = get_correlations(
    model, dataloaders["validation"], device=device, as_dict=False, per_neuron=False
)

print(validation_correlation)

0.1445416
